# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            This lab uses FREE models only: Ollama (local, no API key needed) and Groq (free tier API). Install Ollama from https://ollama.com or get a free Groq API key from https://console.groq.com
            </span>
        </td>
    </tr>
</table>

In [24]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [25]:
# Using FREE models only: Groq (free tier) and Ollama (local, no API key needed)

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"✓ Groq API Key found (begins {groq_api_key[:4]})")
    print("  Using Groq's FREE tier API")
else:
    print("✗ Groq API Key not set")
    print("  Get a free key at: https://console.groq.com")

# Check if Ollama is available
print("\nChecking Ollama (local, free):")
try:
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=5)
    print("✓ Ollama is installed and available")
    print("\nAvailable models:")
    print(result.stdout)
except FileNotFoundError:
    print("✗ Ollama not found. Please install from https://ollama.com")
except Exception as e:
    print(f"Error checking Ollama: {e}")



✓ Groq API Key found (begins gsk_)
  Using Groq's FREE tier API

Checking Ollama (local, free):
✓ Ollama is installed and available

Available models:
NAME                ID              SIZE      MODIFIED    
deepseek-r1:1.5b    e0979632db5a    1.1 GB    11 days ago    
llama3.2:latest     a80c4f17acd5    2.0 GB    2 weeks ago    
gemma3:1b           8648f39daa8f    815 MB    2 weeks ago    



In [26]:
# Connect to FREE services only

# Groq (free tier API)
groq_url = "https://api.groq.com/openai/v1"
if groq_api_key:
    groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
    print("✓ Connected to Groq API (free tier)")
else:
    groq = None
    print("✗ Groq not available (no API key)")

# Ollama (local, free)
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
print("✓ Connected to Ollama at", ollama_url)



✓ Connected to Groq API (free tier)
✓ Connected to Ollama at http://localhost:11434/v1


In [27]:
# FREE models only!

# Groq models (free tier API - very fast!)
GROQ_LLAMA_70B = "llama-3.3-70b-versatile"
GROQ_LLAMA_8B = "llama-3.1-8b-instant"

# Ollama models (local, free - check what you have installed)
OLLAMA_MODELS = [
    "llama3.2:latest",
    "deepseek-r1:1.5b",
    "gemma3:1b",
]

# Build models list and clients dict
models = []
clients = {}

if groq:
    models.extend([GROQ_LLAMA_70B, GROQ_LLAMA_8B])
    clients[GROQ_LLAMA_70B] = groq
    clients[GROQ_LLAMA_8B] = groq

models.extend(OLLAMA_MODELS)
for model in OLLAMA_MODELS:
    clients[model] = ollama

print(f"\nAvailable FREE models ({len(models)} total):")
for model in models:
    print(f"  - {model}")


Available FREE models (5 total):
  - llama-3.3-70b-versatile
  - llama-3.1-8b-instant
  - llama3.2:latest
  - deepseek-r1:1.5b
  - gemma3:1b


In [28]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': False,
 'rustc': {'path': '',
  'version': '',
  'host_triple': '',
  'release': '',
  'commit_hash': ''},
 'cargo': {'path': '', 'version': ''},
 'rustup': {'path': '',
  'version': '',
  'active_toolchain': '',
  'default_toolchain': '',
  'toolchains': [],
  'targets_installed': []},
 'rust_analyzer': {'path': ''},
 'env': {'CARGO_HOME': '',
  'RUSTUP_HOME': '',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': []}

In [29]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

# Use the first available model's client
client = clients[models[0]]
response = client.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

You need to install a Rust toolchain to compile and run Rust code. Here are the simplest step-by-step instructions:

1. **Install Rust toolchain using `rustup`**:
   - Go to the [Rust installation page](https://www.rust-lang.org/tools/install) and follow the installation instructions for Windows.
   - Alternatively, you can use the command line to install `rustup` by running the following command in PowerShell as an administrator:
     ```powershell
     Set-ExecutionPolicy RemoteSigned -Scope CurrentUser
     Invoke-Expression (New-Object System.Net.WebClient).DownloadString('https://get.rustup.rs')
     ```

2. **Verify the Rust installation**:
   - Restart your terminal or command prompt.
   - Run the following command to verify that Rust is installed:
     ```bash
     rustc --version
     ```

To compile and execute your `main.rs` file using Python, you can use the following commands:

* **Compile command**: `rustc -O3 main.rs` (The `-O3` flag enables the maximum possible runtime performance)
* **Run command**: `./main.exe` (On Windows, the compiled executable will have the same name as the source file but with a `.exe` extension)

Here are the commands in markdown:
```markdown
compile_command = "rustc -O3 main.rs"
run_command = "./main.exe"
```
So, your Python code should look like this:
```python
import subprocess

compile_command = "rustc -O3 main.rs"
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

run_command = "./main.exe"
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

return run_result.stdout
```

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [30]:
compile_command = [
    "/Users/ed/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main",
]

run_command = ["./main"]


## And now, on with the main task

In [31]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [32]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [33]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [34]:
def port(model, python):
    client = clients[model]
    response = client.chat.completions.create(model=model, messages=messages_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [35]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [36]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [37]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [38]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


## Your Experiment Results

Test the FREE models and record your Rust execution times:

**Groq Models:**
- Llama 3.3 70B: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds
- Llama 3.1 8B: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds

**Ollama Local Models:**
- llama3.2:latest: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds
- deepseek-r1:1.5b: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds
- gemma3:1b: 
  - Result: ___ (Success/Fail)
  - Rust Time: ___ seconds

**Note:** This is a challenging task - converting Python to Rust is more complex than Python to C++. Don't worry if some models fail!

## About Python-to-Rust Conversion with FREE Models

This is a **challenging exercise** that tests the limits of free models:

**Why Rust?**
- Extremely fast execution (often faster than C++)
- Memory safe without garbage collection
- Complex syntax that's harder for models to generate

**Using FREE Models:**
- **Groq Free Tier:** Fast inference, good for complex tasks
- **Ollama Local:** Smaller models may struggle with Rust syntax

**What to Expect:**
- Some models may fail to generate valid Rust code
- Successful conversions will show impressive speedups
- This demonstrates how challenging Python-to-Rust is compared to Python-to-C++

**Tips:**
- Start with simpler Python code if you get errors
- Check the Rust output for compilation errors
- Install larger Ollama models for better results: `ollama pull qwen2.5-coder:7b`

In [ ]:
# Calculate your speedup!
# If you get successful Rust compilation, compare Python vs Rust times
# Example: If Python takes 0.067s and Rust takes 0.0003s, that's 223x faster!
print("Test the models above and record your results!")

Test the models above and record your results!


Traceback (most recent call last):
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\projects\llm_engineering\.venv\Lib\site-packages\anyio\to_thread.py", line 56, in run_sync
    return await get